In [ ]:
!pip install langchain-google-genai langchain-community chromadb

## API Configuration and Imports

The API key is configured as an environment variable to securely authenticate requests to the LLM service. Required classes for the language model and embedding model are then imported to enable text generation and vector embedding creation.

In [1]:
import os
import warnings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

warnings.filterwarnings("ignore")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Get the API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

## Model Initialization

The Gemini language model and the embedding model are initialized to handle text generation and convert text into vector embeddings for semantic search and retrieval tasks.

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

In [9]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

In [11]:
result = llm.invoke("What is Hyde(Hypothetical Document Embedding)?")
print(result.content)

**Hyde (Hypothetical Document Embedding)** is a technique used in information retrieval and semantic search, particularly within the context of Large Language Models (LLMs) and Retrieval-Augmented Generation (RAG) systems.

The core idea behind Hyde is to address the challenge of matching short, potentially ambiguous user queries to longer, more detailed documents in a vector space.

Here's a breakdown:

### The Problem Hyde Solves

When you perform a semantic search, you convert a user's query into a vector (an embedding) and then compare it to the vectors of all your indexed documents. The closer the vectors, the more semantically similar they are.

However, short user queries often lack the rich context present in the documents you're trying to retrieve. This can lead to:
*   **Suboptimal Embeddings:** A short query like "new policy" might not generate a very specific embedding.
*   **Mismatched Granularity:** A query embedding might not align well with the embeddings of full docume

In [37]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

In [38]:
model_name = "BAAI/bge-small-en-v1.5"
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

In [39]:
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    #model_kwargs={'device': 'cuda'},
    encode_kwargs=encode_kwargs,
)

vector = embedding_function.embed_query("Hello, how are you?")

In [40]:
print(len(vector))

384


In [41]:
print(vector[:5])

[-0.035551004111766815, -0.030710898339748383, 0.027684830129146576, -0.009486943483352661, -0.021329984068870544]


## Data Loading and Splitting
The notebook loads a blog post about prompt engineering and splits it into smaller chunks using a recursive character splitter

In [42]:
# Loading data from the blog link mentioned in the video
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
data = loader.load()


In [43]:
len(data)

1

In [44]:

# Splitting data into chunks of 300 with an overlap of 50
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(data)

In [45]:
len(splits)

217

In [46]:
print(splits[1])

page_content='Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation' metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down l

## Vector Store Setup
The chunks are stored in an in-memory Chroma database


In [47]:
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_function)
retriever = vectorstore.as_retriever()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## HyDE - Hypothetical Document Generation
A prompt is created to ask the LLM to generate a hypothetical answer to a user query before searching the database

In [48]:
# User question
question = "What are the different Chain of Thought prompting?"

# Prompt template for the hypothetical answer
template = """For the given question, try to generate a hypothetical answer. 
Only generate the answer and nothing else.
Question: {question}"""

prompt_hyde = ChatPromptTemplate.from_template(template)

# Generate the hypothetical document
chain_hyde = prompt_hyde | llm
hypothetical_answer = chain_hyde.invoke({"question": question})
print("Hypothetical Document Generated:")
print(hypothetical_answer.content)

Hypothetical Document Generated:
The different Chain of Thought prompting techniques include:

*   **Standard Chain of Thought (CoT) / Few-shot CoT:** Providing a few examples that demonstrate the step-by-step reasoning process.
*   **Zero-shot Chain of Thought (Zero-shot CoT):** Instructing the model to "think step by step" or similar without providing any examples.
*   **Self-consistency Chain of Thought:** Generating multiple diverse CoT paths for a single query and then selecting the most common answer among them.
*   **Tree of Thoughts (ToT):** Exploring multiple reasoning branches and evaluating intermediate thoughts, similar to a search tree.
*   **Graph of Thoughts (GoT):** A more generalized framework than ToT, allowing for arbitrary graph structures to represent and explore thoughts.
*   **Least-to-Most Prompting:** Breaking down a complex problem into a series of simpler sub-problems and solving them sequentially, using the answer to one sub-problem to solve the next.
*   **

## Formatting Context and Final Generation
A function is defined to clean and merge the retrieved documents into a single context string for the final answer

In [50]:
# 1. Retrieval based on Hypothetical Answer
similar_document_1 = retriever.get_relevant_documents(hypothetical_answer.content)
similar_document_1



[Document(metadata={'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.\n\n\nMemory\n\nShort-term memory: I would consider all the in-context learning (See Prompt Engineering) as utiliz

In [52]:
similar_document_1[0].page_content

'Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks'

In [54]:
# 2. Retrieval based on Direct Query (Naive)
similar_document_2 = retriever.get_relevant_documents(question)
similar_document_2

[Document(metadata={'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.\n\n\nMemory\n\nShort-term memory: I would consider all the in-context learning (See Prompt Engineering) as utiliz

In [55]:
similar_document_2[0].page_content

'}\nReferences#\n[1] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022\n[2] Yao et al. “Tree of Thoughts: Dliberate Problem Solving with Large Language Models.” arXiv preprint arXiv:2305.10601 (2023).'

In [59]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

formatted_doc_1 = format_docs(similar_document_1)
formatted_doc_2 = format_docs(similar_document_2)


In [60]:
formatted_doc_1

'Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks\n\nTree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning possibilities at each step. It first decomposes the problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure. The search process can be BFS (breadth-first search) or DFS\n\n}\nReferences#\n[1] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022\n[2] Yao et al. “Tree of Thoughts: Dliberate Problem Solving with Large Language Models.” arXiv preprint arXiv:2305.10601 (2023).\n\nChain of Hindsight (CoH; Liu et al. 2023) encourages the model to improve on its own outputs by explicitly presenting it with a sequence of past out

In [61]:
formatted_doc_2

'}\nReferences#\n[1] Wei et al. “Chain of thought prompting elicits reasoning in large language models.” NeurIPS 2022\n[2] Yao et al. “Tree of Thoughts: Dliberate Problem Solving with Large Language Models.” arXiv preprint arXiv:2305.10601 (2023).\n\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks\n\nThe ReAct prompt template incorporates explicit steps for LLM to think, roughly formatted as:\nThought: ...\nAction: ...\nObservation: ...\n... (Repeated many times)\n\nChain of Hindsight (CoH; Liu et al. 2023) encourages the model to improve on its own outputs by explicitly presenting it with a sequence of past outputs, each annotated with feedback. Human feedback data is a collection of $D_h = \\{(x, y_i , r_i , z_i)\\}_{i=1}^n$, where $x$ i

In [ ]:
# Final prompt template
template_rag = """Please answer the question based on the given context.
Context: {context}
Question: {question}"""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

# Comparison of responses
print("--- Response based on HyDE Retrieval ---")
final_prompt_1 = prompt_rag.format(context=formatted_doc_1, question=question)
response_1 = llm.invoke(final_prompt_1)
print(response_1.content)



--- Response based on HyDE Retrieval ---
Based on the provided context, the different Chain of Thought prompting techniques mentioned are:

1.  **Chain of Thought (CoT)**: This is the standard technique where the model is instructed to "think step by step" to decompose complex tasks into smaller, simpler steps.
2.  **Tree of Thoughts (ToT)**: This technique extends CoT by exploring multiple reasoning possibilities at each step. It decomposes a problem into multiple thought steps and generates multiple thoughts per step, creating a tree structure.

Chain of Hindsight (CoH) is also mentioned, but the context does not describe it as a type of Chain of Thought prompting; rather, it's presented as a technique for improving model outputs using feedback.


In [57]:
print("\n--- Response based on Naive Retrieval ---")
final_prompt_2 = prompt_rag.format(context=formatted_doc_2, question=question)
response_2 = llm.invoke(final_prompt_2)
print(response_2.content)


--- Response based on Naive Retrieval ---
Based on the given context, the different Chain of Thought prompting techniques mentioned are:

*   **Chain of Thought (CoT)**: This is a standard prompting technique where the model is instructed to "think step by step" to decompose complex tasks into smaller, simpler steps.
*   **ReAct prompt template**: This template incorporates explicit steps for the LLM to think, roughly formatted as "Thought: ... Action: ... Observation: ..." (repeated). This is a specific structure for implementing the "think step by step" idea.


## Complete HyDE Pipeline

```text
User Question
      │
      ▼
LLM generates Hypothetical Document
      │
      ▼
Embedding Model
      │
      ▼
Vector Database
      │
      ▼
Retriever
      │
      ▼
Retrieved Documents
      │
      ▼
format_docs()
      │
      ▼
One Context String
      │
      ▼
Prompt Template
      │
      ▼
LLM
      │
      ▼
Final Answer
```

## Comparison of HyDE Retrieval vs Naive Retrieval

In this notebook, the final section compares two retrieval approaches:

### HyDE Retrieval
- The retriever searches using the **LLM-generated hypothetical document**.
- The retrieved documents are then provided to the **LLM** to generate the final answer.

### Naive Retrieval
- The retriever searches using the **original user question**.
- The retrieved documents are then provided to the **LLM** to generate the final answer.

### Conclusion
This comparison helps evaluate whether **HyDE Retrieval** retrieves more relevant context and produces a more accurate and informative final answer than **Naive Retrieval**.